# 02 — Option Chain Scraper

Scrape the full option chain snapshot (OI, IV, Volume, Bid/Ask, LTP) for all strikes.

In [1]:
%run 01_setup.ipynb

Project root: d:\nse-options-scraper
Session: 2026-10-07 (Wed) | now 2026-10-08 00:55 IST | 2026-10-08 is not a finished trading session
Watchlist: 211 symbols | Active (Sprint 1): 2


In [2]:
# Shared XPaths and helpers come from 01_setup.ipynb
from src.session_guard import scrape_preflight, reject_if_stale

def download_option_chain(driver, symbol, item, expiry_text, is_front=False):
    symbol_type = item.get("type", item.get("asset_class", "index"))
    dropdown = driver.find_element(By.XPATH, XPATH_EXPIRY_DROPDOWN)
    Select(dropdown).select_by_visible_text(expiry_text)
    wait_for_option_chain_table(driver, symbol_type)

    spot = get_underlying_spot(driver)
    before = set(os.listdir(TEMP_DOWNLOAD))

    download_btn = find_download_csv_button(driver, timeout=20)
    safe_click(driver, download_btn)

    try:
        downloaded = wait_for_download(before, timeout=30)
    except TimeoutError as e:
        print(f"    [WARN] Download timeout for {expiry_text}: {e}")
        return None

    expiry_clean = expiry_text.replace(" ", "-").replace(",", "")
    dest_dir = os.path.join(DATA_RAW, symbol.lower(), "option_chain")
    os.makedirs(dest_dir, exist_ok=True)
    dest_file = os.path.join(dest_dir, f"{TODAY}_{expiry_clean}.csv")

    shutil.move(downloaded, dest_file)
    print(f"    Saved: {dest_file}")

    meta = {
        "date": TODAY,
        "symbol": symbol,
        "asset_class": item.get("asset_class", item["type"]),
        "index_name": item.get("index_name"),
        "expiry_type": item.get("expiry_type"),
        "expiry": expiry_text,
        "underlying_spot": spot,
        "source_file": dest_file,
        "scraped_at_ist": now_ist().isoformat(timespec="seconds"),
    }
    save_chain_metadata(dest_file, meta, is_front=is_front)
    reject_if_stale(dest_file, TODAY)
    regime = meta.get("expiry_regime", "?")
    print(f"    Expiry tag: dte={meta.get('dte')} {regime} front={meta.get('is_front_expiry')} ({meta.get('signal_regime')})")
    canary_check_chain(dest_file)
    return dest_file

In [3]:
# ── RUN (Sprint 1: NIFTY + BANKNIFTY only) ───────
FORCE_RESCRAPE = False  # True overwrites chains already saved as final for this session

if scrape_preflight(DATA_RAW, [i["symbol"] for i in get_active_watchlist()], TODAY, force=FORCE_RESCRAPE):
    driver = get_driver()
    clear_temp_downloads()

    try:
        init_nse_session(driver)

        for item in get_active_watchlist():
            symbol = item["symbol"]
            symbol_type = item["type"]
            print(f'\n{"="*50}')
            print(f"  {symbol_type.upper()}: {symbol}")
            print(f'{"="*50}')

            select_symbol(driver, symbol, symbol_type)
            expiries = get_expiries_for_symbol(driver, symbol, symbol_type)

            for i, expiry in enumerate(expiries):
                print(f"  Downloading chain for expiry: {expiry}")
                download_option_chain(driver, symbol, item, expiry, is_front=(i == 0))
                slow_delay()

        print(f"\nDone! Chains for session {TODAY} saved under {DATA_RAW}")

    except Exception as e:
        print(f"Error: {e}")
        raise

    finally:
        driver.quit()
        print("Browser closed.")

Visiting NSE homepage for session cookies...
Navigating to Option Chain page...
Option chain page loaded.

  INDEX: NIFTY
  Selected index: NIFTY
  Using 6 expiries for NIFTY: ['13-Oct-2026', '19-Oct-2026', '27-Oct-2026', '03-Nov-2026', '09-Nov-2026', '23-Nov-2026']
    Saved: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_13-Oct-2026.csv
    Expiry tag: dte=6 T-6 front=True (front_month)
    Canary [PASS]: 108 data rows in 2026-10-07_13-Oct-2026.csv
    Saved: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_19-Oct-2026.csv
    Expiry tag: dte=12 T-8+ front=False (back_series)
    Canary [PASS]: 104 data rows in 2026-10-07_19-Oct-2026.csv
    Saved: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_27-Oct-2026.csv
    Expiry tag: dte=20 T-8+ front=False (back_series)
    Canary [PASS]: 144 data rows in 2026-10-07_27-Oct-2026.csv
    Saved: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_03-Nov-2026.csv
    Expiry tag: dte=27 T-8+ f